# 05 — Spark Structured Streaming (4 JP)

**Pelatihan Pengolahan Data dengan Spark — TA 2026**

Data latihan bersifat sintetis, bukan data wajib pajak sebenarnya.


**Day 3: 2 JP; day 4: 2 JP. Theory 2 JP; practice 2 JP.** File-based source uses no additional runtime connector and is suitable for an offline classroom once images are built. Kafka remains available for an optional advanced demo; Kafka Spark connector JARs are **not** bundled.

Before running streaming, on the host execute `python3 scripts/reset_stream.py` and then, after starting the query, `python3 scripts/emit_stream.py --count 20 --interval 2`. Do not rerun the same checkpoint with changed query definitions.


In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
    .appName("SparkTraining-Notebook")
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4050")
    .config("spark.blockManager.port", "4051")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
print("Spark:", spark.version, "master:", spark.sparkContext.master)


In [ ]:
schema=T.StructType([
 T.StructField("event_id",T.StringType()),T.StructField("taxpayer_id",T.StringType()),
 T.StructField("event_time",T.StringType()),T.StructField("amount",T.LongType()),
 T.StructField("channel",T.StringType())])
source=(spark.readStream.schema(schema).option("maxFilesPerTrigger",2)
    .json("/home/spark/stream-input").withColumn("ts",F.to_timestamp("event_time")))
print("Streaming?",source.isStreaming)


In [ ]:
windowed=(source.withWatermark("ts","2 minutes")
    .groupBy(F.window("ts","1 minute"),"channel")
    .agg(F.count("*").alias("events"),F.sum("amount").alias("total")))
query=(windowed.writeStream.outputMode("update")
    .format("memory").queryName("payment_windows_training")
    .option("checkpointLocation","/home/spark/outputs/stream_checkpoint_notebook")
    .start())
print("Started:",query.id)


In [ ]:
# Run this cell repeatedly after emitting new files.
spark.sql("SELECT * FROM payment_windows_training ORDER BY window.start, channel").show(50,False)
print("Active:",query.isActive,"Last progress:",query.lastProgress)


In [ ]:
# STOP the query before leaving or rerunning earlier cells.
query.stop()
print("Stopped")


### Lab tasks
1. Explain event-time vs processing-time.
2. Change window duration to 2 minutes **after clearing checkpoint**.
3. Explain late records and how watermark affects state retention/finalization.
4. Compare `append`, `update`, and `complete` modes and sink limitations.
5. Bonus: write a non-aggregated stream to Parquet with a separate checkpoint.

**Deliverable:** streaming demo and notes on watermark/out-of-order event limitations.
